CNN Architecture

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import pandas as pd
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.init as init

class ExperimentalCNN(nn.Module):
    def __init__(self, num_classes, activation_type='relu', init_type='kaiming'):
        super(ExperimentalCNN, self).__init__()

        if activation_type == 'relu':
            self.activation = nn.ReLU()
        elif activation_type == 'tanh':
            self.activation = nn.Tanh()
        elif activation_type == 'leaky_relu':
            self.activation = nn.LeakyReLU(0.01)

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            self.activation,
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            self.activation,
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            self.activation,
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((4, 4))

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            self.activation,
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

        self._initialize_weights(init_type)

    def _initialize_weights(self, init_type):
        for m in self.modules():
            if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
                if init_type == 'xavier':
                    init.xavier_uniform_(m.weight)
                elif init_type == 'kaiming':
                    init.kaiming_normal_(m.weight, nonlinearity='relu')
                else:
                    init.normal_(m.weight, mean=0, std=0.01)
                if m.bias is not None:
                    init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = self.classifier(x)
        return x

Optimizer

In [4]:
def get_optimizer(model, opt_type, lr=0.001):
    if opt_type == 'sgd':
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif opt_type == 'adam':
        return optim.Adam(model.parameters(), lr=lr)
    elif opt_type == 'rmsprop':
        return optim.RMSprop(model.parameters(), lr=lr)

def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    return running_loss / len(train_loader)

Data Processing

In [5]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Loading CIFAR-10
train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

100%|██████████| 170M/170M [00:18<00:00, 9.05MB/s]


Train

In [6]:
def run_experiment(dataset_name, train_loader, val_loader, num_classes):
    activations = ['relu', 'tanh', 'leaky_relu']
    initializations = ['xavier', 'kaiming', 'random']
    optimizers = ['sgd', 'adam', 'rmsprop']

    results = []

    print(f"\n{'='*60}")
    print(f"STARTING EXPERIMENT ON: {dataset_name}")
    print(f"{'='*60}")
    print(f"{'Activation':<12} | {'Init':<10} | {'Optimizer':<10} | {'Accuracy':<8} | {'Loss':<8}")
    print("-" * 60)

    for act in activations:
        for init in initializations:
            for opt_name in optimizers:
                model = ExperimentalCNN(num_classes=num_classes, activation_type=act, init_type=init).to(device)

                optimizer = get_optimizer(model, opt_name)
                criterion = nn.CrossEntropyLoss()

                epochs = 10
                for epoch in range(epochs):
                    _ = train_model(model, train_loader, criterion, optimizer, device)

                val_acc, val_loss = evaluate_model(model, val_loader, criterion, device)

                print(f"{act:<12} | {init:<10} | {opt_name:<10} | {val_acc:>7.2f}% | {val_loss:>8.4f}")

                results.append({
                    'activation': act,
                    'init': init,
                    'optimizer': opt_name,
                    'accuracy': val_acc,
                    'loss': val_loss
                })

    df = pd.DataFrame(results)
    df.to_csv(f"experiment_results_{dataset_name}.csv", index=False)
    print(f"\n{'='*60}")
    print(f"Experiment Complete. Results saved to experiment_results_{dataset_name}.csv")
    return df

Evaluate

In [7]:
def evaluate_model(model, val_loader, criterion, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    avg_loss = running_loss / len(val_loader)
    return accuracy, avg_loss

Implementation

In [8]:
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split

# CIFAR-10
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
cifar_test = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

train_size = int(0.8 * len(cifar_train))
val_size = len(cifar_train) - train_size
cifar_train_subset, cifar_val_subset = random_split(cifar_train, [train_size, val_size])

cifar_loaders = {
    'train': DataLoader(cifar_train_subset, batch_size=64, shuffle=True),
    'val': DataLoader(cifar_val_subset, batch_size=64, shuffle=False)
}

# Cats vs Dogs
cvd_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


In [ ]:
print("Starting CIFAR-10 Grid Search...")
cifar_results_df = run_experiment(
    dataset_name="CIFAR10",
    train_loader=cifar_loaders['train'],
    val_loader=cifar_loaders['val'],
    num_classes=10
)

best_custom_config = cifar_results_df.loc[cifar_results_df['accuracy'].idxmax()]
print(f"Best Custom CNN Accuracy: {best_custom_config['accuracy']:.2f}%")

Starting CIFAR-10 Grid Search...

STARTING EXPERIMENT ON: CIFAR10
Activation   | Init       | Optimizer  | Accuracy | Loss    
------------------------------------------------------------
relu         | xavier     | sgd        |   69.52% |   0.8607
relu         | xavier     | adam       |   72.89% |   0.7790
relu         | xavier     | rmsprop    |   70.74% |   0.8607
relu         | kaiming    | sgd        |   67.49% |   0.9227


Comparing with Renet

In [ ]:
from torchvision import models

def run_resnet_baseline(dataset_name, train_loader, val_loader, num_classes):
    print(f"\nRunning Pretrained ResNet-18 Baseline on {dataset_name}...")

    # Load and modify ResNet-18
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    model = model.to(device)

    # Standard fine-tuning setup
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Train and evaluate
    for epoch in range(5): # Fewer epochs needed for transfer learning
        train_model(model, train_loader, criterion, optimizer, device)

    res_acc, res_loss = evaluate_model(model, val_loader, criterion, device)
    print(f"ResNet-18 Final Results -> Accuracy: {res_acc:.2f}%, Loss: {res_loss:.4f}")

    return res_acc, res_loss